# 持久执行
持久执行是一种技术，流程或工作流会在关键点保存进度，使其能够暂停并在之后从中断处准确恢复。这在需要人机交互的场景中尤其有用，在这些场景中，用户可以在继续之前检查、验证或修改流程；在长时间运行的任务中，这些任务可能会遇到中断或错误（例如，对 LLM 的调用超时）。通过保存已完成的工作，持久执行使流程即使在较长的延迟（例如一周后）之后也能恢复，而无需重新处理之前的步骤。

LangGraph 内置的持久层为工作流提供持久执行，确保每个执行步骤的状态都保存到持久存储中。此功能可确保如果工作流中断（无论是由于系统故障还是人机交互），都可以从上次记录的状态恢复。

## 要求¶
为了利用 LangGraph 中的持久执行，您需要：

- 通过指定将保存工作流进度的检查点来实现工作流的持久性。
- 执行工作流时指定线程标识符。这将跟踪工作流特定实例的执行历史记录。
- 将任何非确定性操作（例如，随机数生成）或具有副作用的操作（例如，文件写入、API 调用）包装在任务内部，以确保在恢复工作流时，这些操作不会在特定运行中重复，而是从持久层检索其结果。有关更多信息，请参阅确定性和一致性重放。

### 确定性和一致性重放¶
恢复工作流程运行时，代码不会从执行停止的同一行代码继续执行；相反，它会确定一个合适的起始点，从中断处继续执行。这意味着工作流程将从起始点重新执行所有步骤，直到到达停止点。

因此，当您编写用于持久执行的工作流时，必须将任何非确定性操作（例如，随机数生成）和任何具有副作用的操作（例如，文件写入、API 调用）包装在任务或节点内。

为了确保您的工作流程具有确定性并且可以一致地重播，请遵循以下准则：

- 避免重复工作：如果一个节点包含多个具有副作用的操作（例如，日志记录、文件写入或网络调用），请将每个操作包装在一个单独的任务中。这确保在恢复工作流时，操作不会重复，并且其结果可以从持久层检索。
- 封装非确定性操作：将任何可能产生非确定性结果（例如，随机数生成）的代码封装在任务或节点中。这确保了工作流在恢复时，能够完全按照记录的步骤顺序执行，并产生相同的结果。
使用幂等操作：尽可能确保副作用（例如 API 调用、文件写入）具有幂等性。这意味着，如果在工作流失败后重试某个操作，其效果将与首次执行时相同。这对于导致数据写入的操作尤为重要。如果某个任务已启动但未能成功完成，工作流的恢复将重新运行该任务，并依赖记录的结果来保持一致性。使用- 幂等键或验证现有结果以避免意外重复，从而确保工作流执行顺畅且可预测。

有关需要避免的一些陷阱示例，请参阅函数式 API 中的“常见陷阱”部分，其中展示了如何使用任务来构建代码以避免这些问题。同样的原则也适用于StateGraph (Graph API)。

## 在节点中使用任务¶
如果一个节点包含多个操作，您可能会发现将每个操作转换为一个任务比将操作重构为单个节点更容易。


In [ ]:
### 原来
from typing import NotRequired
from typing_extensions import TypedDict
import uuid

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, START, END
import requests

# Define a TypedDict to represent the state
class State(TypedDict):
    url: str
    result: NotRequired[str]

def call_api(state: State):
    """Example node that makes an API request."""
    result = requests.get(state['url']).text[:100]  # Side-effect
    return {
        "result": result
    }

# Create a StateGraph builder and add a node for the call_api function
builder = StateGraph(State)
builder.add_node("call_api", call_api)

# Connect the start and end nodes to the call_api node
builder.add_edge(START, "call_api")
builder.add_edge("call_api", END)

# Specify a checkpointer
checkpointer = InMemorySaver()

# Compile the graph with the checkpointer
graph = builder.compile(checkpointer=checkpointer)

# Define a config with a thread ID.
thread_id = uuid.uuid4()
config = {"configurable": {"thread_id": thread_id}}

# Invoke the graph
graph.invoke({"url": "https://www.example.com"}, config)

In [ ]:
#### 修改后的
from typing import NotRequired
from typing_extensions import TypedDict
import uuid

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.func import task
from langgraph.graph import StateGraph, START, END
import requests

# Define a TypedDict to represent the state
class State(TypedDict):
    urls: list[str]
    result: NotRequired[list[str]]


@task
def _make_request(url: str):
    """Make a request."""
    return requests.get(url).text[:100]

def call_api(state: State):
    """Example node that makes an API request."""
    requests = [_make_request(url) for url in state['urls']]
    results = [request.result() for request in requests]
    return {
        "results": results
    }

# Create a StateGraph builder and add a node for the call_api function
builder = StateGraph(State)
builder.add_node("call_api", call_api)

# Connect the start and end nodes to the call_api node
builder.add_edge(START, "call_api")
builder.add_edge("call_api", END)

# Specify a checkpointer
checkpointer = InMemorySaver()

# Compile the graph with the checkpointer
graph = builder.compile(checkpointer=checkpointer)

# Define a config with a thread ID.
thread_id = uuid.uuid4()
config = {"configurable": {"thread_id": thread_id}}

# Invoke the graph
graph.invoke({"urls": ["https://www.example.com"]}, config)


## 恢复工作流程¶
在工作流中启用持久执行后，您可以针对以下场景恢复执行：

- 暂停和恢复工作流：使用中断函数在特定点暂停工作流，并使用命令原语以更新后的状态恢复工作流。更多详情，请参阅“人机交互” 。
- 从故障中恢复：发生异常（例如 LLM 提供程序中断）后，自动从上一个成功的检查点恢复工作流。这需要通过向工作流提供 aNone作为输入值，以相同的线程标识符执行该工作流（请参阅此使用函数式 API 的示例）。

## 恢复工作流程的起点¶
- 如果您使用的是StateGraph（Graph API），则起点是执行停止的节点的开始处。
- 如果您在节点内部进行子图调用，则起始点将是调用已暂停子图的父节点。在子图内部，起始点将是执行停止的具体节点。
- 如果您使用的是功能 API，则起点是执行停止的入口点的开头。